#### Getting started With Langchain And Open AI

In this quickstart we'll see how to:

- Get setup with LangChain, LangSmith and LangServe
- Use the most basic and common components of LangChain: prompt templates, models, and output parsers.
- Build a simple application with LangChain
- Trace your application with LangSmith
- Serve your application with LangServe

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
## Langsmith Tracking
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

In [2]:
import langchain
print(langchain.__version__)

1.2.15


In [3]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="gpt-4o-mini")
print(llm)

client=<openai.resources.chat.completions.completions.Completions object at 0x7bc1d42b41d0> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7bc1d4bbb380> root_client=<openai.OpenAI object at 0x7bc2d007c380> root_async_client=<openai.AsyncOpenAI object at 0x7bc1d427e7e0> model_name='gpt-4o-mini' model_kwargs={} openai_api_key=SecretStr('**********')


In [4]:
## Input and get response form LLM

result=llm.invoke("What is generative AI?")

In [5]:
result.__dict__

{'content': 'Generative AI refers to a class of artificial intelligence techniques that can create new content, such as text, images, music, and other forms of media, based on the patterns and information it has learned from existing data. Unlike traditional AI, which primarily focuses on analyzing and classifying data, generative AI can produce original outputs that resemble human-created content.\n\nKey characteristics and components of generative AI include:\n\n1. **Models and Algorithms**: Generative AI often uses machine learning models, particularly deep learning architectures like Generative Adversarial Networks (GANs), Variational Autoencoders (VAEs), and transformer models. These models learn to generate new data by understanding the underlying structure of the training data.\n\n2. **Training Data**: Generative AI systems are trained on large datasets that contain examples of the kind of content they are expected to generate. For instance, a text generation model might be trai

In [6]:
### Chatprompt Template
from langchain_core.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are an expert AI Engineer. Provide me answers based on the questions"),
        ("user","{input}")
    ]

)
prompt

ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are an expert AI Engineer. Provide me answers based on the questions'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [7]:
## chain
chain=prompt|llm
response=chain.invoke({"input":"what is langsmith and Can you tell me about Langsmith?"})

In [8]:
print(response)

content='Langsmith is a platform designed to help developers and teams build, manage, and optimize their AI applications, particularly those that leverage large language models (LLMs). It offers tools and features that facilitate the prompt engineering process, allowing users to create and refine prompts that guide the behavior of AI models to achieve desired outputs.\n\nKey features of Langsmith typically include:\n\n1. **Prompt Management**: Users can create, store, and organize prompts for different use cases, making it easier to maintain and iterate on AI interactions.\n\n2. **Version Control**: Langsmith may offer versioning capabilities, enabling teams to track changes to prompts over time and revert to previous versions if needed.\n\n3. **Collaboration Tools**: The platform often includes features that support collaboration among team members, allowing them to share prompts, provide feedback, and work together on AI projects.\n\n4. **Testing and Optimization**: Langsmith allows 

In [11]:
print(response.response_metadata)

{'token_usage': {'completion_tokens': 261, 'prompt_tokens': 38, 'total_tokens': 299, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a7190374f3', 'finish_reason': 'stop', 'logprobs': None}


In [12]:
type(response)

langchain_core.messages.ai.AIMessage

In [12]:
## stroutput Parser

from langchain_core.output_parsers import StrOutputParser
output_parser=StrOutputParser()
chain=prompt|llm|output_parser

response=chain.invoke({"input":"Can you tell me about Langsmith?"})
print(response)

As of my last update in October 2023, Langsmith is a platform designed to assist developers and teams in building, deploying, and managing AI applications, particularly those that involve natural language processing (NLP). It aims to streamline the process of integrating AI capabilities into software products, making it easier for users to implement language models and other AI features without needing extensive expertise in machine learning.

Langsmith often focuses on providing tools for model management, data handling, and user interface design specifically tailored for language-based applications. This can include features like versioning for language models, monitoring performance, and optimizing deployment for various environments.

If you are looking for specific features, use cases, or updates beyond October 2023, I would recommend checking Langsmith's official website or recent announcements for the latest information.


In [13]:
## SimpleJsonOutputParser Parser

from langchain_core.output_parsers import SimpleJsonOutputParser
output_parser=SimpleJsonOutputParser()
chain=prompt|llm|output_parser

response=chain.invoke({"input":"Can you tell me about Langsmith and give the data in json format"})
print(response)

{'name': 'Langsmith', 'description': 'A platform for developing and deploying AI applications, focusing on natural language processing.', 'features': {'model_development': {'description': 'Tools for building and training language models.', 'capabilities': ['Custom model creation', 'Pre-trained model integration', 'Fine-tuning support']}, 'testing': {'description': 'Capabilities for testing and evaluating models.', 'capabilities': ['Automated testing frameworks', 'Evaluation metrics and reporting', 'A/B testing capabilities']}, 'deployment': {'description': 'Easily deploy language models to production.', 'capabilities': ['API creation for models', 'Containerization support', 'Monitoring and logging']}, 'collaboration': {'description': 'Facilitate team collaboration on AI projects.', 'capabilities': ['Version control for models', 'Project management tools', 'Documentation support']}, 'integration': {'description': 'Seamless integration with existing workflows.', 'capabilities': ['Support

In [14]:
## PydanticOutputParser - Define schema and parser

from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

# Define the expected output schema
class LangsmithInfo(BaseModel):
    name: str = Field(description="Name of the tool")
    description: str = Field(description="Short description of the tool")
    features: list[str] = Field(description="List of key features")

# Create parser from Pydantic model
parser = JsonOutputParser(pydantic_object=LangsmithInfo)

# Auto-generated format instructions to inject into the prompt
print(parser.get_format_instructions())


STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Markdown in your output):


In [25]:
## Use format instructions in prompt + chain

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert AI Engineer. Answer the question.\n{format_instructions}"),
    ("user", "{input}")
]).partial(format_instructions=parser.get_format_instructions())

chain = prompt | llm | parser

response = chain.invoke({"input": "Tell me about Langsmith"})
print(response)

{'name': 'Langsmith', 'description': 'A collaborative platform for building AI applications and tools.', 'features': ['Real-time collaboration', 'Version control for AI models', 'Integration with popular programming languages', 'Comprehensive documentation', 'User-friendly interface']}
